# Scotland Weather Forecast Synthesiser

This notebook scrapes several specialist forecast sources for Scotland and uses GPT to synthesise them into a single, clear summary covering:

- **Snow & Mountain conditions** (Highlands & Cairngorms)
- **Surf** (Thurso East — Scotland's most famous wave)
- **Rain & General Weather**
- **Wind**

Run each cell top-to-bottom with `Shift+Enter`.

> **Note:** Some weather sites use JavaScript rendering and may return limited content with this simple scraper. The notebook handles that gracefully — if a source returns little data the LLM will say so.

In [1]:
# imports

import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# Load environment variables and check the key

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found - check your .env file!")
elif not api_key.startswith("sk-proj-"):
    print("API key found but doesn't start with sk-proj- — double-check it.")
else:
    print("API key found and looks good!")

API key found and looks good!


In [3]:
openai = OpenAI()

## Scraper

We define a local scraper here that grabs a bit more text per source (5,000 chars vs the 2,000 limit in `scraper.py`) so the LLM has richer input to synthesise from.

In [4]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_forecast(url, char_limit=5_000):
    """Fetch plain text from a URL, stripped of scripts/styles, up to char_limit."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.title.string.strip() if soup.title else "No title"
        for tag in soup(["script", "style", "img", "input", "nav", "footer"]):
            tag.decompose()
        body = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
        return (title + "\n\n" + body)[:char_limit]
    except Exception as e:
        return f"[Could not fetch {url}: {e}]"

"## Forecast Sources\n",
    "\n",
    "| Source | Covers |\n",
    "|--------|--------|\n",
    "| MWIS Highlands & Cairngorms | Snow, mountain weather |\n",
    "| MWIS Western Highlands | Snow, mountain weather |\n",
    "| XCWeather Scotland | Wind, rain, general |\n",
    "| Met Office Scotland overview | Rain, wind, general |\n",
    "| **Surf** | Chosen below — pick as many spots as you like |"

In [5]:
BASE_SOURCES = {
    "Mountain & Snow — Cairngorms & Monadhliath (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/cairngorms-np-and-monadhliath/text",
    "Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/west-highlands/text",
    "Mountain & Snow — Northwest Highlands (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/the-northwest-highlands/text",
    "Wind & Rain — Scotland (XCWeather)":
        "https://www.xcweather.co.uk/forecast/scotland",
    "General — Met Office Scotland":
        "https://www.metoffice.gov.uk/weather/forecast/gcpvj0v07",
}

## Choose Your Surf Spots\n\nAll known Scottish surf breaks are listed in `SURF_SPOTS` below.\nEdit `selected_surf_spots` to include whichever ones you want in the briefing.

In [6]:
# All available Scottish surf spots
SURF_SPOTS = {
    "Thurso East":    "https://www.surf-forecast.com/breaks/Thurso-East/forecasts/latest",
    "Belhaven Bay":   "https://www.surf-forecast.com/breaks/Belhaven-Bay/forecasts/latest",
    "Pease Bay":      "https://www.surf-forecast.com/breaks/Pease-Bay/forecasts/latest",
    "Machrihanish":   "https://www.surf-forecast.com/breaks/Machrihanish/forecasts/latest",
    "Scarista":       "https://www.surf-forecast.com/breaks/Scarista/forecasts/latest",
    "Tiree (The Maze)": "https://www.surf-forecast.com/breaks/The-Maze/forecasts/latest",
    "Coldingham Bay": "https://www.surf-forecast.com/breaks/Coldingham-Bay/forecasts/latest",
    "Sandend Bay":    "https://www.surf-forecast.com/breaks/Sandend-Bay/forecasts/latest",
}

print("Available surf spots:")
for name in SURF_SPOTS:
    print(f"  - {name}")

Available surf spots:
  - Thurso East
  - Belhaven Bay
  - Pease Bay
  - Machrihanish
  - Scarista
  - Tiree (The Maze)
  - Coldingham Bay
  - Sandend Bay


In [7]:
# ✏️  Edit this list to choose which surf spots to include
selected_surf_spots = [
    "Thurso East",
    "Belhaven Bay",
]

# Build the final forecast sources dict from base sources + selected surf spots
FORECAST_SOURCES = dict(BASE_SOURCES)
for spot in selected_surf_spots:
    if spot in SURF_SPOTS:
        FORECAST_SOURCES[f"Surf — {spot}"] = SURF_SPOTS[spot]
    else:
        print(f"⚠️  '{spot}' not found in SURF_SPOTS — check the name and try again.")

print("Forecast sources for this run:")
for name in FORECAST_SOURCES:
    print(f"  • {name}")

Forecast sources for this run:
  • Mountain & Snow — Cairngorms & Monadhliath (MWIS)
  • Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)
  • Mountain & Snow — Northwest Highlands (MWIS)
  • Wind & Rain — Scotland (XCWeather)
  • General — Met Office Scotland
  • Surf — Thurso East
  • Surf — Belhaven Bay


## Fetch All Forecasts

This cell scrapes each source and prints how many characters were retrieved. If a site uses heavy JavaScript you'll see a low character count — that's normal with a plain requests scraper.

In [8]:
raw_forecasts = {}

for name, url in FORECAST_SOURCES.items():
    print(f"Fetching: {name}...")
    content = fetch_forecast(url)
    raw_forecasts[name] = content
    print(f"  → {len(content):,} characters retrieved\n")

Fetching: Mountain & Snow — Cairngorms & Monadhliath (MWIS)...
  → 5,000 characters retrieved

Fetching: Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)...
  → 5,000 characters retrieved

Fetching: Mountain & Snow — Northwest Highlands (MWIS)...
  → 5,000 characters retrieved

Fetching: Wind & Rain — Scotland (XCWeather)...
  → 1,594 characters retrieved

Fetching: General — Met Office Scotland...
  → 5,000 characters retrieved

Fetching: Surf — Thurso East...
  → 5,000 characters retrieved

Fetching: Surf — Belhaven Bay...
  → 189 characters retrieved



## Prompts

We give the model a system prompt that sets it up as a Scottish weather expert, then a user prompt that bundles all the raw forecast text for synthesis.

In [9]:
system_prompt = """
You are an expert Scottish weather analyst who knows the Highlands, the Cairngorms,
and the north coast surf scene well. You synthesise raw forecast data from multiple
sources into one clear, readable briefing.

Structure your response in markdown with these sections:
1. **Overview** — one-paragraph summary of the overall picture
2. **Snow & Mountain Conditions** — avalanche risk, snow depth, visibility, wind chill
3. **Surf** — swell height, period, wind direction, quality rating
4. **Rain & Wind** — precipitation amounts, wind speeds and directions, any warnings
5. **Outlook** — brief 3-5 day trend

If a section has no usable data (e.g. the site was JavaScript-rendered and returned
nothing meaningful), say so briefly rather than making things up.
Respond in markdown. Do not wrap the markdown in a code block.
"""

In [10]:
def build_user_prompt(forecasts):
    prompt = "Please synthesise the following raw forecast data for Scotland into a structured briefing.\n\n"
    for source_name, content in forecasts.items():
        prompt += f"---\n### Source: {source_name}\n{content}\n\n"
    return prompt

## Synthesise!

Now we pass all the fetched content to GPT and ask it to produce the unified forecast briefing.

In [11]:
def synthesise_forecasts(forecasts):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_prompt(forecasts)},
        ]
    )
    return response.choices[0].message.content

In [12]:
briefing = synthesise_forecasts(raw_forecasts)
display(Markdown(briefing))

# Scottish Weather Briefing (November 15, 2026)

## Overview
A winter pattern is settling in across Scotland with significant snowfall expected in the Highlands and varying conditions across the coasts. The mountains will see challenging conditions, with strong winds and poor visibility making navigation difficult. Mild temperatures will persist in the far south, while colder air moves in from the north.

## Snow & Mountain Conditions
- **Avalanche Risk**: Moderate risk due to fresh snow accumulation and wind.
- **Snow Depth**: Continuous snowfalls are expected in higher regions, creating challenging conditions. Lower elevations may alternate with rain.
- **Visibility**: Generally poor due to persistent fog and heavy cloud cover, particularly in the Cairngorms and Northwest Highlands.
- **Wind Chill**: Air temperatures will drop below 0°C on higher mountains, with wind chill making it feel like -10°C to -15°C.

## Surf
- **Swell Height**: 
  - Afternoon: 3.0m (10ft) from the west with a 10s period.
  - Evening: 2.4m (8ft) from WNW with a 9s period.
- **Wind Direction**: Cross-onshore winds expected.
- **Quality Rating**: Fair to good for experienced surfers due to solid swell but caution is advised due to stronger rip currents.

## Rain & Wind
- **Precipitation Amounts**: Up to 21mm of rain expected, with heavy showers during the day.
- **Wind Speeds**: Winds ranging from 25 to 40mph across the regions, with gusts potentially reaching 50mph in exposed areas. Winds will be predominantly north-northwest.
- **Weather Warnings**: Blustery conditions are causing patchy fog and limited visibility. A warning for high winds may be issued later as conditions worsen.

## Outlook
Over the next few days:
- **Monday**: Expect some clearer skies and rising temperatures, especially in the south, but wind speeds will pick up later in the day. Chance of snow showers, particularly in northern areas.
- **Tuesday**: Winds will shift easterly and strengthen, potentially leading to severe conditions. Snowfall likely increases again with low temperatures persisting.
- **Midweek**: A transient improvement could occur Wednesday before further weather systems roll in towards the weekend. 

Stay safe and well-prepared for winter conditions as you venture out!

## Try It Yourself

Some ideas to extend this:
- Add a source for sea temperature (useful for surfers and wild swimmers)
- Change the system prompt to output a **WhatsApp-style** message instead of markdown
- Add a specific location (e.g. Ben Nevis, Glencoe, Caithness) and tailor the briefing
- Use `gpt-4o` instead of `gpt-4o-mini` for a more detailed synthesis

In [13]:
# Your turn — add extra sources or tweak the system prompt here

# Example: add a sea temperature source
# extra_sources = {
#     "Sea Temp — Scotland": "https://www.seatemperature.org/europe/united-kingdom/scotland.htm",
# }
# for name, url in extra_sources.items():
#     raw_forecasts[name] = fetch_forecast(url)

# Then re-run: display(Markdown(synthesise_forecasts(raw_forecasts)))